# Week 8 build · LoRA by hand, then with PEFT, then a rank ablation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjtri/learnml/blob/main/notebooks/week-08.ipynb)

**~3 hours · T4 for Part E (Runtime → Change runtime type → T4) · run top to bottom.**

**The one rule:** every experiment is preceded by a 🔮 **predict cell**. Type what you expect and how sure you are *before* running the experiment. The notebook refuses empty predictions. A rough guess is fine; the gap between guess and result is what you are here to collect (canvas steps 7–8: *predict before running*, *log the surprise*).

**Two passes.** First run with `SMOKE = True` (next cell but one): Parts A to D on a tiny sensor-glove model, CPU only, under five minutes, so you see every cell work and the rank ablation once. Then set `SMOKE = False`, switch to the T4, and run again: Part E LoRA-tunes SmolLM2-360M for real (about 30 min of training).

| Part | Topic | Time |
|---|---|---|
| A | Rank and SVD in numbers: a nearly low-rank matrix, an image at k = 1, 4, 16 | 25 min |
| B | `LoRALinear` by hand; adapt a frozen glove-gesture model to a new user | 35 min |
| C | The same through PEFT: match the count, check the weight merge | 20 min |
| D | Rank ablation r ∈ {1, 4, 16} × 2 seeds; plant forgetting and leakage | 35 min |
| E | T4: LoRA-tune SmolLM2-360M on instruction rows; before/after; the same ablation | 60 min |
| F | Stretch: adapter on one layer only | 15 min |

In [ ]:
#@title Setup: run me first
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display
torch.manual_seed(0); np.random.seed(0)
_LEDGER = {}

def predict(tag, prediction, confidence):
    """Log a prediction BEFORE the experiment. Refuses empty predictions."""
    if not str(prediction).strip():
        raise ValueError(f"[{tag}] Write a prediction first. A rough guess is fine; an empty one is not.")
    _LEDGER[tag] = {"prediction": str(prediction).strip(), "confidence": int(confidence), "actual": "", "surprise": ""}
    print(f"Logged [{tag}] at {confidence}% confidence: {prediction}")

def record(tag, what_happened, surprise="none"):
    if tag not in _LEDGER:
        raise ValueError(f"[{tag}] has no prediction yet. Run its predict cell first.")
    _LEDGER[tag].update(actual=str(what_happened).strip(), surprise=surprise)
    print(f"Recorded [{tag}] surprise={surprise}")

def reveal():
    rows = ["| # | prediction | sure | what happened | surprise |", "|---|---|---|---|---|"]
    rows += [f"| {t} | {r['prediction']} | {r['confidence']}% | {r['actual']} | {r['surprise']} |" for t, r in _LEDGER.items()]
    print("\n".join(rows)); display(Markdown("\n".join(rows)))

print("torch", torch.__version__, "| ready")

---
## Part A · Rank and SVD in numbers (25 min)

**Words used in this part.** *rank*: how many independent directions a matrix's outputs fill; `torch.linalg.matrix_rank` counts singular values above a tolerance. *singular value*: the strength of one stretch in the SVD, sorted largest first. *rank-k approximation*: keep the first k stretches only. *relative error*: size of what was dropped over size of the whole.

In [ ]:
#@title Config: SMOKE = True is a five-minute CPU rehearsal; False adds the real T4 run in Part E
SMOKE = True  #@param {type:"boolean"}
import math, time, json, os, hashlib, copy, torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = dict(n_in=16, n_latent=8, hidden=128, n_classes=10, noise=0.8, n_pre=512, n_user=64, n_test=256,
           pre_steps=400 if SMOKE else 1500, ft_steps=200 if SMOKE else 400, lr=3e-3, seed=0)
lm = dict(name="HuggingFaceTB/SmolLM2-360M", n_train=400, n_test=50, steps=300, abl_steps=150,
          r=8, alpha=16, lr=2e-4, batch=4, max_len=256, targets=["q_proj", "v_proj"])
RUN_LM = (not SMOKE) and device == "cuda"
if not SMOKE and device == "cpu":
    print("WARNING: SMOKE=False needs a GPU for Part E. Runtime -> Change runtime type -> T4.")
torch.manual_seed(cfg["seed"])
print("device:", device, "| SMOKE:", SMOKE, "| Part E will run:", RUN_LM)

In [ ]:
#@title 🔮 Predict A1: W = randn(64,3) @ randn(3,64), then W_noisy = W + 0.01·noise. matrix_rank of each? How many of W_noisy's 64 singular values are 'big'?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A1", prediction, confidence)

In [ ]:
torch.manual_seed(0)
W = torch.randn(64, 3) @ torch.randn(3, 64)                 # rank 3 by construction
W_noisy = W + 0.01 * torch.randn(64, 64)
print("rank(W)       =", torch.linalg.matrix_rank(W).item())
print("rank(W_noisy) =", torch.linalg.matrix_rank(W_noisy).item(), "  <- floating point sees every direction")
S = torch.linalg.svdvals(W_noisy)
print("singular values, top 6:", [round(v, 2) for v in S[:6].tolist()])
print("rank with tolerance 1.0:", torch.linalg.matrix_rank(W_noisy, atol=1.0).item())
print("numbers needed: full", W.numel(), "| rank-3 factors", 3 * (64 + 64))

In [ ]:
#@title ✍️ A1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A1", what_happened, surprise)

Surprise? Adding noise of size 0.01 made the matrix *full rank*: rank is exact arithmetic, and floating point has no zeros. What you want is the sorted singular values: three big ones, then a floor. "Nearly low rank" is a statement about that plot, which is why lesson 2's tool is the one that matters in practice.

In [ ]:
#@title 🔮 Predict A2: Render the letters 'ML' as a 48×48 image and keep k = 1, 4, 16 stretches. At which k is it readable? Relative error at k = 4 (percent)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("A2", prediction, confidence)

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt, numpy as np
fig = plt.figure(figsize=(1, 1), dpi=48); fig.text(0.5, 0.5, "ML", ha="center", va="center", fontsize=28, weight="bold")
fig.canvas.draw(); rgba = np.asarray(fig.canvas.buffer_rgba()); plt.close(fig)
img = torch.tensor(1 - rgba[:, :, 0] / 255.0, dtype=torch.float32)     # 48×48, 1 = ink
U, S, Vh = torch.linalg.svd(img, full_matrices=False)
fig, axes = plt.subplots(1, 4, figsize=(8, 2.2))
axes[0].imshow(img, cmap="gray_r"); axes[0].set_title("original 2304 numbers")
for ax, k in zip(axes[1:], [1, 4, 16]):
    img_k = U[:, :k] @ torch.diag(S[:k]) @ Vh[:k]
    err = torch.linalg.norm(img - img_k) / torch.linalg.norm(img)
    ax.imshow(img_k, cmap="gray_r"); ax.set_title(f"k={k}: {k * 97} numbers, err {100 * err:.0f}%")
    print(f"k={k:2d}  stored {k * (48 + 48 + 1):5d} ({100 * k * 97 / 2304:4.1f}%)  relative error {100 * err:5.1f}%")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
print("first 8 singular values / largest:", [round(v, 3) for v in (S[:8] / S[0]).tolist()])

In [ ]:
#@title ✍️ A2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("A2", what_happened, surprise)

---
## Part B · LoRA by hand on a glove model (35 min)

**The story.** A data glove has 16 sensor channels. Ten hand gestures are encoded by an 8-number latent code, and each user's glove fit is a fixed 16×8 mixing matrix from code to sensors; the sensors are noisy. The base model is trained on **user A**. **User B** wears the glove differently (the mixing matrix is rotated), so the base model fails on B. You get 64 labelled samples from B, "ten minutes of paired data", and may only train a small adapter. This is the `bridge-adaptation` card: the learner is the frozen model, the device mapping is the adapter.

**Words used in this part.** *frozen*: `requires_grad=False`, no gradient, no optimizer state. *`LoRALinear`*: a frozen linear layer plus a trainable B·A change, B starting at zero. *trainable count*: `sum(p.numel() for p in params if p.requires_grad)`. *accuracy*: fraction of test gestures classified correctly, on samples never trained on.

In [ ]:
g = torch.Generator().manual_seed(cfg["seed"])
PROTO = torch.randn(cfg["n_classes"], cfg["n_latent"], generator=g) * 1.5     # one latent code per gesture
M_A = torch.randn(cfg["n_in"], cfg["n_latent"], generator=g) / math.sqrt(cfg["n_latent"])
R, _ = torch.linalg.qr(torch.randn(cfg["n_latent"], cfg["n_latent"], generator=g))   # a random rotation
M_B = M_A @ R + 0.1 * torch.randn(cfg["n_in"], cfg["n_latent"], generator=g)          # user B's glove fit

def make_data(M, n, seed):
    gen = torch.Generator().manual_seed(seed)
    y = torch.randint(0, cfg["n_classes"], (n,), generator=gen)
    x = PROTO[y] @ M.T + cfg["noise"] * torch.randn(n, cfg["n_in"], generator=gen)
    return x.to(device), y.to(device)

xA, yA = make_data(M_A, cfg["n_pre"], 1);   xA_test, yA_test = make_data(M_A, cfg["n_test"], 2)
xB, yB = make_data(M_B, cfg["n_user"], 3);  xB_test, yB_test = make_data(M_B, cfg["n_test"], 4)

class MLP(nn.Module):
    def __init__(self, n_in, hidden, n_out):
        super().__init__()
        self.fc1, self.fc2, self.head = nn.Linear(n_in, hidden), nn.Linear(hidden, hidden), nn.Linear(hidden, n_out)
    def forward(self, x):
        return self.head(F.relu(self.fc2(F.relu(self.fc1(x)))))

print("user A train", tuple(xA.shape), "| user B labelled", tuple(xB.shape), "| tests", tuple(xB_test.shape))

In [ ]:
def train(model, params, x, y, steps, lr, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(params, lr=lr)
    for _ in range(steps):
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

@torch.no_grad()
def acc(model, x, y):
    return (model(x).argmax(1) == y).float().mean().item()

def n_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

torch.manual_seed(cfg["seed"])
base = MLP(cfg["n_in"], cfg["hidden"], cfg["n_classes"]).to(device)
final = train(base, base.parameters(), xA, yA, cfg["pre_steps"], cfg["lr"])
base.requires_grad_(False)                                   # from here on, W is frozen
print(f"pretrained on user A: final loss {final:.3f}, parameters {sum(p.numel() for p in base.parameters()):,}")

In [ ]:
#@title 🔮 Predict B1: The base model's accuracy on user A's test set, and on user B's test set (two numbers, chance is 0.10).
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B1", prediction, confidence)

In [ ]:
print(f"base model  | user A test acc {acc(base, xA_test, yA_test):.2f} | user B test acc {acc(base, xB_test, yB_test):.2f}")

In [ ]:
#@title ✍️ B1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B1", what_happened, surprise)

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer, r):
        super().__init__()
        self.base = base_layer.requires_grad_(False)                       # frozen W
        self.A = nn.Parameter(torch.randn(r, base_layer.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base_layer.out_features, r))     # starts at zero: BA = 0
    def forward(self, x):
        return self.base(x) + x @ self.A.T @ self.B.T                      # x meets A first: r wide

def add_lora(model, r, layers=("fc1", "fc2")):
    m = copy.deepcopy(model)
    for name in layers:
        setattr(m, name, LoRALinear(getattr(m, name), r))
    return m

def lora_params(model):
    return [p for p in model.parameters() if p.requires_grad]

In [ ]:
#@title 🔮 Predict B2: r = 4 on fc1 (16→128) and fc2 (128→128): how many numbers train, and what fraction of the base model's parameter count is that?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B2", prediction, confidence)

In [ ]:
lora4 = add_lora(base, r=4)
hand = 4 * (16 + 128) + 4 * (128 + 128)
print("trainable by formula r(in+out) per layer:", hand, "| by counting:", n_trainable(lora4))
print(f"fraction of the base model: {100 * hand / sum(p.numel() for p in base.parameters()):.1f}%")
with torch.no_grad():
    print("outputs identical to the base before training:", torch.allclose(lora4(xB_test), base(xB_test)))

In [ ]:
#@title ✍️ B2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B2", what_happened, surprise)

In [ ]:
#@title 🔮 Predict B3: Fine-tune on user B's 64 samples: LoRA r=4 versus full fine-tuning of every weight. Accuracy on B test for each, and on A test afterwards (forgetting)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B3", prediction, confidence)

In [ ]:
rows = []
lora4 = add_lora(base, r=4)
train(lora4, lora_params(lora4), xB, yB, cfg["ft_steps"], cfg["lr"])
rows.append(("LoRA r=4", n_trainable(lora4), acc(lora4, xB_test, yB_test), acc(lora4, xA_test, yA_test)))
full = copy.deepcopy(base).requires_grad_(True)
train(full, full.parameters(), xB, yB, cfg["ft_steps"], cfg["lr"])
rows.append(("full fine-tune", n_trainable(full), acc(full, xB_test, yB_test), acc(full, xA_test, yA_test)))
rows.append(("base (no tuning)", 0, acc(base, xB_test, yB_test), acc(base, xA_test, yA_test)))
print(f"{'run':18s} {'trainable':>9s} {'B test':>7s} {'A test':>7s}")
for name, n, b, a in rows:
    print(f"{name:18s} {n:9,d} {b:7.2f} {a:7.2f}")

In [ ]:
#@title ✍️ B3: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B3", what_happened, surprise)

Surprise? Read the two columns separately. *B test* is what the adaptation bought; *A test* is what it cost. Full fine-tuning has 12× more trainable numbers and only 64 samples to learn them from; the adapter cannot move far from the base, which is both its limit and its protection. Part D asks how much of this depends on r.

---
## Part C · The same through PEFT (20 min)

**Words used in this part.** *`LoraConfig`*: r, `lora_alpha`, `lora_dropout`, `target_modules` by name. *`get_peft_model`*: wraps every named module with a LoRA layer, in place. *`print_trainable_parameters`*: the line to check against your hand count. *`merge_and_unload`*: fold B·A into W once and return a plain model. With `lora_alpha = r` the scale α/r is 1, the same as the hand version.

In [ ]:
#@title 🔮 Predict C1: PEFT with r=4 on fc1 and fc2: does print_trainable_parameters match your hand count from B2 exactly? Roughly what will it print for 'all params'?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C1", prediction, confidence)

In [ ]:
from peft import LoraConfig, get_peft_model
peft_cfg = LoraConfig(r=4, lora_alpha=4, lora_dropout=0.0, target_modules=["fc1", "fc2"])
peft4 = get_peft_model(copy.deepcopy(base), peft_cfg)
peft4.print_trainable_parameters()
n_peft, n_all = peft4.get_nb_trainable_parameters()
print("hand count from B2:", hand, "| PEFT:", n_peft, "| match:", n_peft == hand)
with torch.no_grad():
    print("PEFT outputs identical to the base before training:", torch.allclose(peft4(xB_test), base(xB_test)))
train(peft4, lora_params(peft4), xB, yB, cfg["ft_steps"], cfg["lr"])
print(f"PEFT r=4 after training | B test {acc(peft4, xB_test, yB_test):.2f} | A test {acc(peft4, xA_test, yA_test):.2f}")

In [ ]:
#@title ✍️ C1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict C2: After merge_and_unload: the largest output difference between the merged plain model and the PEFT model? Its parameter count compared with the base?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C2", prediction, confidence)

In [ ]:
with torch.no_grad():
    before = peft4(xB_test)
merged = peft4.merge_and_unload()                       # W + B·A computed once; LoRA layers removed
with torch.no_grad():
    after = merged(xB_test)
print("largest |merged - unmerged| output difference:", (after - before).abs().max().item())
print("merged model parameters:", sum(p.numel() for p in merged.parameters()),
      "| base:", sum(p.numel() for p in base.parameters()))
print("layer types now:", type(merged.fc1).__name__, type(merged.fc2).__name__)

In [ ]:
#@title ✍️ C2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C2", what_happened, surprise)

Surprise? The difference is floating-point noise and the count is back to the base's: merging changes layout, not numbers. Keep the adapter file anyway; a merged model can no longer swap adapters.

---
## Part D · Rank ablation, then two planted failures (35 min)

**Words used in this part.** *ablation*: change one thing (r), keep everything else fixed, read the difference. *seed spread*: the gap between two runs that differ only in the random seed; a rank effect smaller than this is not an effect. *forgetting*: A-test accuracy after tuning on B. *leakage*: a test sample that also sits in the training set.

In [ ]:
#@title 🔮 Predict D1: r ∈ {1, 4, 16}, seeds {0, 1}, same steps. Write the ordering of B-test accuracy you expect, and whether the gaps between ranks will be larger than the gap between the two seeds.
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D1", prediction, confidence)

In [ ]:
results = {}
for r in [1, 4, 16]:
    for seed in [0, 1]:
        torch.manual_seed(seed)                       # the seed sets A's random start
        m = add_lora(base, r=r)
        train(m, lora_params(m), xB, yB, cfg["ft_steps"], cfg["lr"], seed=seed)
        results[(r, seed)] = (acc(m, xB_test, yB_test), acc(m, xA_test, yA_test), n_trainable(m))
print(f"{'r':>3s} {'seed':>4s} {'trainable':>9s} {'B test':>7s} {'A test':>7s}")
for (r, seed), (b, a, n) in results.items():
    print(f"{r:3d} {seed:4d} {n:9,d} {b:7.2f} {a:7.2f}")
print()
for r in [1, 4, 16]:
    bs = [results[(r, s)][0] for s in [0, 1]]
    print(f"r={r:2d}: B test mean {sum(bs) / 2:.2f}, seed spread {abs(bs[0] - bs[1]):.2f}")

In [ ]:
#@title ✍️ D1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D1", what_happened, surprise)

**Reading it honestly.** Two seeds give a spread, not an error bar; a rank gap inside that spread has not been shown. Read the *A test* column too: what r buys on B it tends to cost on A, the trade lesson 5 called forgetting. In this toy the change user B needs is a rotation of an 8-number code, so the required rank is about 8 *by construction*; the LoRA paper's r = 1 result is about language models, where the needed change turned out far lower-rank than the matrices. Same tool, different fact about the data.

In [ ]:
#@title 🔮 Predict D2: Copy ONE user-B test sample into the 64 training samples, then train LoRA r=16. Its loss afterwards, versus the mean loss of the other 255 test samples?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D2", prediction, confidence)

In [ ]:
h = lambda t: hashlib.sha1(t.detach().cpu().numpy().tobytes()).hexdigest()
seen = {h(t) for t in xB}
print(len([t for t in xB_test if h(t) in seen]), "test items also in train   <- the check; must be 0")
x_leak, y_leak = torch.cat([xB, xB_test[:1]]), torch.cat([yB, yB_test[:1]])   # plant one leak on purpose
seen = {h(t) for t in x_leak}
print(len([t for t in xB_test if h(t) in seen]), "test items also in train   <- after planting")
m = add_lora(base, r=16)
train(m, lora_params(m), x_leak, y_leak, 3 * cfg["ft_steps"], cfg["lr"])
with torch.no_grad():
    per_item = F.cross_entropy(m(xB_test), yB_test, reduction="none")
print(f"loss on the leaked item {per_item[0]:.3f} | mean loss on the other {len(per_item) - 1}: {per_item[1:].mean():.3f}")
print(f"items with loss below the leaked one: {(per_item[1:] < per_item[0]).sum().item()} of {len(per_item) - 1}")

In [ ]:
#@title ✍️ D2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D2", what_happened, surprise)

Surprise? The leaked item sits at a loss of about zero, but so do the easy items of its class, so on a toy one item's loss is weak evidence. In Part E every training row is unique text, and a leaked row's loss drops far below its neighbours: that is what leakage looks like at scale. The hash check costs nothing and is the line to keep.

---
## Part E · The real thing: LoRA-tune SmolLM2-360M on a T4 (60 min)

**Runs only when `SMOKE = False` and a GPU is present** (`RUN_LM` above). Everything in this part is guarded, so in the rehearsal these cells print one line and move on. The predict cells still work; write your predictions on the real pass.

**Words used in this part.** *instruction rows*: (instruction, response) pairs from the Dolly dataset's "brainstorming" category, 400 for training and 50 kept unseen. *base-text loss*: the model's loss on a fixed plain paragraph, before and after, the forgetting meter. *`week08_runs.jsonl`*: one line per run with `cfg`, `seed`, `r`, and the numbers.

In [ ]:
if RUN_LM:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "datasets"], check=False)
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from datasets import load_dataset
    tok = AutoTokenizer.from_pretrained(lm["name"])
    tok.pad_token = tok.pad_token or tok.eos_token
    def load_base():
        return AutoModelForCausalLM.from_pretrained(lm["name"], dtype=torch.float32).to(device)   # T4: no bf16
    base_lm = load_base()
    print(f"{lm['name']}: {sum(p.numel() for p in base_lm.parameters()) / 1e6:.0f}M parameters")
else:
    print("skipped: set SMOKE = False on a T4 runtime to run Part E")

In [ ]:
if RUN_LM:
    ds = load_dataset("databricks/databricks-dolly-15k", split="train").filter(lambda r: r["category"] == "brainstorming")
    ds = ds.shuffle(seed=0).select(range(lm["n_train"] + lm["n_test"]))
    fmt = lambda r: f"### Instruction:\n{r['instruction']}\n### Response:\n{r['response']}{tok.eos_token}"
    texts = [fmt(r) for r in ds]
    train_texts, test_texts = texts[:lm["n_train"]], texts[lm["n_train"]:]
    h = lambda t: hashlib.sha1(t.encode()).hexdigest()
    seen = {h(t) for t in train_texts}
    print(len([t for t in test_texts if h(t) in seen]), "test items also in train   <- must be 0")
    print(train_texts[0][:300])
else:
    print("skipped")

In [ ]:
PARAGRAPH = ("The river rises in the hills and runs east through farmland before it reaches the city. "
             "In spring the water is high and brown with silt; by late summer it is slow and clear. "
             "People fish from the bridges in the evening, and the ferries stop running at dusk.")
if RUN_LM:
    def batch_of(rows):
        enc = tok(rows, return_tensors="pt", padding=True, truncation=True, max_length=lm["max_len"]).to(device)
        labels = enc["input_ids"].clone(); labels[enc["attention_mask"] == 0] = -100
        return dict(enc, labels=labels)
    @torch.no_grad()
    def mean_loss(model, rows, bs=8):
        model.eval(); losses = [model(**batch_of(rows[i:i + bs])).loss.item() for i in range(0, len(rows), bs)]
        model.train(); return sum(losses) / len(losses)
    @torch.no_grad()
    def per_row_loss(model, rows):
        model.eval(); out = [model(**batch_of([r])).loss.item() for r in rows]; model.train(); return out
    before = dict(unseen=mean_loss(base_lm, test_texts), base_text=mean_loss(base_lm, [PARAGRAPH]))
    print("BEFORE | unseen-row loss %.3f | base-text loss %.3f" % (before["unseen"], before["base_text"]))
else:
    print("skipped")

In [ ]:
#@title 🔮 Predict E1: After 300 LoRA steps (r=8, q_proj+v_proj): unseen-row loss compared with BEFORE, and base-text loss compared with BEFORE (two directions and rough sizes)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E1", prediction, confidence)

In [ ]:
def train_lora_lm(r, seed, steps, alpha=None, targets=None, leak=None):
    torch.manual_seed(seed)
    model = get_peft_model(load_base(), LoraConfig(r=r, lora_alpha=alpha or 2 * r, lora_dropout=0.05,
                                                   target_modules=targets or lm["targets"], task_type="CAUSAL_LM"))
    rows = train_texts + ([leak] if leak else [])
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lm["lr"])
    for step in range(steps):
        idx = torch.randint(len(rows), (lm["batch"],)).tolist()
        loss = model(**batch_of([rows[i] for i in idx])).loss
        loss.backward(); opt.step(); opt.zero_grad()
        if step % 50 == 0: print(f"step {step:4d} train loss {loss.item():.3f}")
    return model
if RUN_LM:
    tuned = train_lora_lm(lm["r"], 0, lm["steps"], alpha=lm["alpha"])
    tuned.print_trainable_parameters()
    after = dict(unseen=mean_loss(tuned, test_texts), base_text=mean_loss(tuned, [PARAGRAPH]))
    print("AFTER  | unseen-row loss %.3f | base-text loss %.3f" % (after["unseen"], after["base_text"]))
    with open("week08_runs.jsonl", "a") as f:
        f.write(json.dumps(dict(cfg=lm, seed=0, r=lm["r"], before=before, after=after, when=time.strftime("%Y-%m-%d %H:%M"))) + "\n")
else:
    print("skipped")

In [ ]:
#@title ✍️ E1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E1", what_happened, surprise)

In [ ]:
if RUN_LM:
    prompt = test_texts[0].split("### Response:")[0] + "### Response:\n"
    for name, model in [("base", base_lm), ("LoRA", tuned)]:
        model.eval()
        out = model.generate(**tok(prompt, return_tensors="pt").to(device), max_new_tokens=60, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        print(f"--- {name} ---\n" + tok.decode(out[0][-60:], skip_special_tokens=True).strip()[:400] + "\n")
    print("--- reference response ---\n" + test_texts[0].split("### Response:")[1].strip()[:300])
else:
    print("skipped")

In [ ]:
#@title 🔮 Predict E2: Plant one leaked test row and retrain: its loss versus the median of the other 49 unseen rows?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E2", prediction, confidence)

In [ ]:
if RUN_LM:
    del tuned; torch.cuda.empty_cache()
    leaky = train_lora_lm(lm["r"], 0, lm["steps"], alpha=lm["alpha"], leak=test_texts[0])
    losses = per_row_loss(leaky, test_texts)
    others = sorted(losses[1:])
    print(f"leaked row loss {losses[0]:.3f} | median of the other {len(others)}: {others[len(others) // 2]:.3f} | lowest other: {others[0]:.3f}")
    del leaky; torch.cuda.empty_cache()
else:
    print("skipped")

In [ ]:
#@title ✍️ E2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E2", what_happened, surprise)

In [ ]:
#@title 🔮 Predict E3: Ablation on the real model: r ∈ {1, 4, 16}, seeds {0, 1}, 150 steps each. Ordering of unseen-row loss? Will the rank gaps beat the seed spread? Which r forgets least (base-text loss)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E3", prediction, confidence)

In [ ]:
if RUN_LM:
    table = {}
    for r in [1, 4, 16]:
        for seed in [0, 1]:
            m = train_lora_lm(r, seed, lm["abl_steps"])
            table[(r, seed)] = dict(unseen=mean_loss(m, test_texts), base_text=mean_loss(m, [PARAGRAPH]),
                                    trainable=m.get_nb_trainable_parameters()[0])
            with open("week08_runs.jsonl", "a") as f:
                f.write(json.dumps(dict(cfg=lm, seed=seed, r=r, steps=lm["abl_steps"], **table[(r, seed)])) + "\n")
            del m; torch.cuda.empty_cache()
    print(f"{'r':>3s} {'seed':>4s} {'trainable':>10s} {'unseen':>7s} {'base-text':>9s}")
    for (r, seed), v in table.items():
        print(f"{r:3d} {seed:4d} {v['trainable']:10,d} {v['unseen']:7.3f} {v['base_text']:9.3f}")
    for r in [1, 4, 16]:
        u = [table[(r, s)]["unseen"] for s in [0, 1]]
        print(f"r={r:2d}: unseen mean {sum(u) / 2:.3f}, seed spread {abs(u[0] - u[1]):.3f}")
else:
    print("skipped")

In [ ]:
#@title ✍️ E3: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E3", what_happened, surprise)

**Reading the real table.** Write the ordering you predicted next to the one you got. A rank gap smaller than the seed spread is noise until more seeds say otherwise. Then look at the base-text column: the r that forgets least is usually the smallest r that still moved the unseen loss, and that is the r to use.

---
## Part F · Stretch: adapter on one layer only (15 min)

Back on the tiny model (runs on CPU). Lesson 4's rule was "more modules at a small rank beats one module at a large rank". Test it here, where you might *think* you know where the needed change lives: the glove fit changed at the input, so the first layer.

**Words used in this part.** *target modules*: which layers get an adapter; the others stay frozen.

In [ ]:
#@title 🔮 Predict F1: r=4 on fc1 only, on fc2 only, and on both: which reaches the best B-test accuracy, and does any of them forget less on A?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("F1", prediction, confidence)

In [ ]:
print(f"{'layers':12s} {'trainable':>9s} {'B test':>7s} {'A test':>7s}")
for layers in [("fc1",), ("fc2",), ("fc1", "fc2")]:
    m = add_lora(base, r=4, layers=layers)
    train(m, lora_params(m), xB, yB, cfg["ft_steps"], cfg["lr"])
    print(f"{'+'.join(layers):12s} {n_trainable(m):9,d} {acc(m, xB_test, yB_test):7.2f} {acc(m, xA_test, yA_test):7.2f}")

In [ ]:
#@title ✍️ F1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("F1", what_happened, surprise)

Surprise? Where the adapter attaches matters as much as r, and the intuition "the change lives at the input" can lose: the second layer is wider (128×128 against 16×128), so the same r buys more trainable numbers and more room there. For a language model nobody knows the right attach point in advance, which is why the paper had to measure it (table 5).

---
## Wrap up

Run the cell below, copy the table into your insight ledger next to `lin-lora` (and the failure-modes line next to `bridge-adaptation`), and mark the biggest surprise. Then on the laptop:

```
python track.py done week-08/build --rating <1-5> --minutes 180 --note "biggest surprise"
python gen_week.py 9
```

In [ ]:
reveal()